# Carrier Allocation - Single File Test

Quick test harness for verifying the allocation sheet extraction logic
on **one Excel file at a time**. Use this before running the full pipeline
to confirm that:

1. The UHC skip check behaves correctly for the file name
2. All matching `{Carrier} Alloc YYYY.MM.DD` worksheets are discovered
3. The header row is detected at the right position
4. The correct number of data rows are extracted (no header metadata, no summary footer)
5. The extracted data looks sensible as a pandas DataFrame

> This notebook does **not** write to a delta table. It only reads and displays.
>
> Each cell that touches the workbook opens it fresh and closes it on the
> way out, so individual cells can be re-run safely.

In [ ]:
# ============================================================
# CONFIGURATION - set the single file to test
# ============================================================

# Full path to the Excel file you want to test.
# In Fabric, this typically looks like:
#   "/lakehouse/default/Files/_02. TO BE TRANSFERRED/SomeFile.xlsx"
# Locally, you can point at anything, e.g.:
#   "test-data/TO BE TRANSFERRED     CIGN MS - Pend Alloc 2025.12.31 SH.xlsx"
# TEST_FILE_PATH = "/lakehouse/default/Files/_02. TO BE TRANSFERRED/SomeFile.xlsx"
TEST_FILE_PATH = "test-data\TO BE TRANSFERRED     CIGN MS - Pend Alloc 2025.12.31 SH.xlsx"

# How many rows to preview from each matched sheet.
PREVIEW_ROWS = 10

In [ ]:
import os
import re
import warnings

from openpyxl import load_workbook
import pandas as pd

# openpyxl prints a noisy "Data Validation extension" UserWarning on every
# workbook open; suppress it.
warnings.filterwarnings(
    "ignore",
    message="Data Validation extension is not supported and will be removed",
)

# Show more columns / wider output for inspection.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

# Stub a no-op log used by the shared helpers below (the test notebook
# does not need full logging configured).
import logging
log = logging.getLogger("carrier_alloc_test")

In [ ]:
# ============================================================
# EXTRACTION LOGIC (mirror of the main pipeline)
# ============================================================

ALLOC_SHEET_RE = re.compile(r"^(.+?)\s+Alloc\s+(\d{4}\.\d{2}\.\d{2})$")
SKIP_CARRIER_CODES = {"UHC"}
HEADER_MARKER = "MEMBER NAME"
MAX_HEADER_SCAN = 50


def _is_valid_xlsx(filename):
    """Return True for real .xlsx files, False for temp/hidden files."""
    return filename.lower().endswith(".xlsx") and not filename.startswith("~$")


def discover_excel_files(base_path, folder_cfg):
    """Return a sorted list of .xlsx file paths in a lakehouse folder."""
    folder = os.path.join(base_path, folder_cfg["path"])
    if not os.path.isdir(folder):
        log.warning("Folder not found: %s", folder)
        return []

    files = []

    if folder_cfg.get("root_only", False):
        for name in os.listdir(folder):
            full = os.path.join(folder, name)
            if os.path.isfile(full) and _is_valid_xlsx(name):
                files.append(full)
    else:
        for dirpath, _, filenames in os.walk(folder):
            for name in filenames:
                if _is_valid_xlsx(name):
                    files.append(os.path.join(dirpath, name))

    return sorted(files)


def should_skip_file(filepath):
    """Return True if the filename contains a carrier code we want to skip."""
    name_upper = os.path.basename(filepath).upper()
    return any(code in name_upper for code in SKIP_CARRIER_CODES)


def find_alloc_sheets(sheetnames):
    """Return (sheet_name, carrier_code, alloc_date) for matching sheets.

    Takes a list of sheet names rather than a workbook object so it can
    be called without keeping the workbook open.
    """
    results = []
    for name in sheetnames:
        m = ALLOC_SHEET_RE.match(name)
        if m:
            results.append((name, m.group(1).strip(), m.group(2)))
    return results


def extract_sheet_data(ws):
    """Detect the header row and extract all data rows from a worksheet.

    Strategy:
        1. Scan rows from the top for "MEMBER NAME" in column A -> header row.
        2. Build unique column names from the header (duplicates get a suffix,
           unnamed columns get a "_col_N" placeholder).
        3. Collect every subsequent row whose column A is non-empty. Empty
           column-A rows are either blank spacers or summary/footer rows,
           so they are safely skipped.

    Returns:
        (header_row_excel, col_names, records)
            header_row_excel: 1-based Excel row number of the header,
                              or None if not found.
            col_names:        list of unique column names.
            records:          list of dicts (one per data row).
    """
    all_rows = list(ws.iter_rows(values_only=True))

    # --- Locate header row ---
    header_idx = None
    for i, row in enumerate(all_rows[:MAX_HEADER_SCAN]):
        if row and row[0] is not None:
            if str(row[0]).strip().upper() == HEADER_MARKER:
                header_idx = i
                break

    if header_idx is None:
        return None, [], []

    header = list(all_rows[header_idx])
    n_cols = len(header)

    # --- Build unique column names (case-insensitive dedup) ---
    # Spark/Delta is case-insensitive by default and enforces it for column
    # names (Parquet is case-sensitive but Delta is case-preserving), so two
    # headers like "CARRIER" and "Carrier" would collide on saveAsTable with
    # COLUMN_ALREADY_EXISTS. Bucket by the lowercase form so the second
    # occurrence picks up a "_2" suffix; original casing is preserved on the
    # first occurrence so existing analytics keep working.
    col_names = []
    seen = {}  # key: lowercase name, value: occurrence count
    for j in range(n_cols):
        raw = header[j]
        base = (
            str(raw).strip()
            if raw is not None and str(raw).strip()
            else f"_col_{j + 1}"
        )
        key = base.lower()
        if key not in seen:
            seen[key] = 1
            col_names.append(base)
        else:
            seen[key] += 1
            col_names.append(f"{base}_{seen[key]}")

    # --- Collect data rows ---
    records = []
    for row in all_rows[header_idx + 1:]:
        if not row or row[0] is None or str(row[0]).strip() == "":
            continue  # skip blank / summary rows

        padded = list(row)[:n_cols]
        padded += [None] * (n_cols - len(padded))
        records.append(dict(zip(col_names, padded)))

    return header_idx + 1, col_names, records  # 1-based Excel row

In [ ]:
# ============================================================
# STEP 1: Open the workbook and list all sheets
# ============================================================

assert os.path.isfile(TEST_FILE_PATH), f"File not found: {TEST_FILE_PATH}"

fname = os.path.basename(TEST_FILE_PATH)
print(f"File:        {fname}")
print(f"Full path:   {TEST_FILE_PATH}")
print(f"Skip (UHC):  {should_skip_file(TEST_FILE_PATH)}")
print()

# Open just to read sheet names, then close immediately. Caching the
# names lets later cells run without holding the workbook open.
wb = load_workbook(TEST_FILE_PATH, read_only=True, data_only=True)
try:
    sheetnames = list(wb.sheetnames)
finally:
    wb.close()

print(f"All worksheets ({len(sheetnames)}):")
for name in sheetnames:
    print(f"  - {name}")

In [ ]:
# ============================================================
# STEP 2: Find sheets matching the allocation pattern
# ============================================================

matches = find_alloc_sheets(sheetnames)

if not matches:
    print("No worksheets match the pattern '{Carrier} Alloc YYYY.MM.DD'")
else:
    print(f"Matched {len(matches)} alloc worksheet(s):")
    for sheet_name, carrier, alloc_date in matches:
        print(f"  - '{sheet_name}'  |  carrier='{carrier}'  |  date='{alloc_date}'")

In [ ]:
# ============================================================
# STEP 3: Extract data from each matched sheet and preview
# ============================================================
# This cell is idempotent: it opens its own workbook handle and closes
# it on the way out, so it can be re-run on its own.

previews = {}

wb = load_workbook(TEST_FILE_PATH, read_only=True, data_only=True)
try:
    for sheet_name, carrier, alloc_date in matches:
        print("=" * 70)
        print(f"Sheet: {sheet_name}")
        print("=" * 70)

        ws = wb[sheet_name]
        header_row, col_names, records = extract_sheet_data(ws)

        if header_row is None:
            print("  Header row NOT found - skipping")
            continue

        print(f"  Header row (Excel):  {header_row}")
        print(f"  Columns:             {len(col_names)}")
        print(f"  Data rows extracted: {len(records)}")

        if records:
            first_name = records[0].get("MEMBER NAME")
            last_name  = records[-1].get("MEMBER NAME")
            print(f"  First member:        {first_name!r}")
            print(f"  Last member:         {last_name!r}")

        pdf = pd.DataFrame(records)

        # Attach metadata columns (same as main pipeline).
        alloc_date_iso = alloc_date.replace(".", "-")
        pdf["_source_file"]  = fname
        pdf["_sheet_name"]   = sheet_name
        pdf["_carrier_code"] = carrier
        pdf["_alloc_date"]   = alloc_date_iso

        previews[sheet_name] = pdf
        print()
finally:
    wb.close()

print(f"Extraction complete. {len(previews)} sheet(s) ready to preview.")

In [ ]:
# ============================================================
# STEP 4: Preview the extracted DataFrames
# ============================================================
# In Fabric, `display(pdf)` gives a nicer interactive table; in plain
# Jupyter, the cell output renders the DataFrame directly.

for sheet_name, pdf in previews.items():
    print(f"\n>>> {sheet_name}  ({len(pdf)} rows x {len(pdf.columns)} cols)\n")
    try:
        display(pdf.head(PREVIEW_ROWS))  # Fabric / Jupyter
    except NameError:
        print(pdf.head(PREVIEW_ROWS))    # fallback for bare Python

In [ ]:
# ============================================================
# STEP 5: (Optional) list all column names per sheet
# ============================================================

for sheet_name, pdf in previews.items():
    print(f"\n>>> {sheet_name}")
    for i, col in enumerate(pdf.columns, start=1):
        print(f"  {i:3d}. {col}")